# Intent Gap — Stage C grading (v3)

LLM-as-judge classification of v2 candidate conversations. Each candidate is presented to Gemini 2.5 Flash with the prompt in Section 3.3 / Appendix A and classified as Intent Gap (A), Refusal (B), Hallucination (C), False Positive (D), or Other (E). For A verdicts, the judge also assigns a taxonomy category.

In [ ]:
!pip install -q google-genai==1.66.0

## API client

Reads `GEMINI_API_KEY` from environment / Colab Secrets. Pinned google-genai version avoids upstream Colab dependency conflicts.

In [ ]:
import os
from google import genai

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
client = genai.Client(api_key=GEMINI_API_KEY)

## Load v2 candidates

In [ ]:
import json

candidates = []
with open('pilot_examples_v2.jsonl', 'r') as f:
    for line in f:
        candidates.append(json.loads(line))

## Judge prompt and grading function

In [ ]:
import re
import time

JUDGE_PROMPT = (
    'You are an expert evaluator deciding whether an AI assistant exchange exhibits an '
    'INTENT GAP failure mode.\n\n'
    'CONTEXT:\n'
    'ORIGINAL USER PROMPT:\n{user_prompt}\n\n'
    'ASSISTANT RESPONSE:\n{asst_response}\n\n'
    'USER FOLLOW-UP (which contained the phrase "{matched_phrase}"):\n{user_repair}\n\n'
    'TASK 1: Pick exactly one classification:\n'
    'A. INTENT GAP -- assistant addressed the literal prompt but missed what the user actually wanted.\n'
    'B. REFUSAL -- assistant declined or said it could not help.\n'
    'C. HALLUCINATION -- assistant gave a factually wrong answer.\n'
    'D. FALSE POSITIVE -- user follow-up is not actually a repair.\n'
    'E. OTHER -- none of the above.\n\n'
    'TASK 2: If A (INTENT GAP), pick the BEST taxonomy category:\n'
    '- implicit-context-blindness, specificity-mismatch, format-mismatch, goal-collapse, '
    'sycophantic-drift, silent-failure, refusal-mismatch, memory-state-failure, tone-misread, '
    'cultural-misread, expertise-mismatch, other\n\n'
    'Reply in JSON only:\n'
    '{{"verdict": "A|B|C|D|E", "category": "<id or null>", "note": "<= 20 words"}}'
)

JSON_RE = re.compile(r'\{.*?\}', flags=re.DOTALL)


def grade(c: dict) -> dict:
    prompt = JUDGE_PROMPT.format(
        user_prompt=(c.get('prev_user_prompt') or '')[:1500],
        asst_response=(c.get('prev_asst_response') or '')[:1500],
        user_repair=(c.get('repair_turn') or '')[:1500],
        matched_phrase=c.get('matched_phrase', ''),
    )
    try:
        r = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        raw = r.text or ''
        m = JSON_RE.search(raw)
        if m:
            return json.loads(m.group(0))
    except Exception as e:
        return {'verdict': 'ERR', 'category': None, 'note': str(e)[:200]}
    return {'verdict': 'PARSE_ERR', 'category': None, 'note': 'no JSON in response'}

## Grade candidates

Free-tier rate limit is approximately 15 requests per minute; the sleep below stays well under it.

In [ ]:
from collections import Counter

graded = []
verdict_counts = Counter()
category_counts = Counter()

for i, c in enumerate(candidates):
    verdict = grade(c)
    record = {**c, 'verdict': verdict}
    graded.append(record)
    verdict_counts[verdict.get('verdict', 'ERR')] += 1
    if verdict.get('verdict') == 'A' and verdict.get('category'):
        category_counts[verdict['category']] += 1
    if (i + 1) % 10 == 0:
        print(f'{i + 1}/{len(candidates)}    {dict(verdict_counts)}')
    time.sleep(4.5)

## Serialize

In [ ]:
summary = {
    'n_graded': len(graded),
    'verdicts': dict(verdict_counts),
    'categories': dict(category_counts),
    'judge': 'gemini-2.5-flash',
    'sdk': 'google-genai==1.66.0',
    'note': 'Stage C LLM-as-judge auto-labeling. Hand-spot-check on a stratified subset is reserved for revision.',
}

with open('graded_v3.jsonl', 'w') as f:
    for g in graded:
        f.write(json.dumps(g) + '\n')

with open('graded_v3_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)